# XGBoost on Abalone — pipeline pattern walkthrough

This notebook implements the same hardened pipeline architecture as `01_secure_pipeline.ipynb`
using XGBoost on the Abalone dataset — a simple regression problem that runs in minutes,
not hours.

**Use this notebook to:**
- Learn the SageMaker Pipeline pattern quickly without waiting for Autopilot
- Demonstrate the FailStep governance layer in a live client or YouTube demo
- Verify the hardened infrastructure is working before running the full Autopilot pipeline

**For the production-grade fintech pipeline, see:** `01_secure_pipeline.ipynb`

---

**Pipeline stages:**
1. `ProcessingStep` — download Abalone, feature engineering, train/test split
2. `TrainingStep` — XGBoost regression (~5 minutes)
3. `QualityCheckStep` — RMSE threshold check
4. `ConditionStep` — gate: passes only if RMSE is below threshold
5. `ModelStep` — register to Model Registry (pass branch)
6. `FailStep` — log rejection reason and halt (fail branch)

**Prerequisites:** Run `make deploy` before executing this notebook.

## 1. Setup

In [ ]:
import boto3
import sagemaker
import subprocess
import urllib.request
import os

from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.workflow.functions import Join
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.quality_check_step import (
    QualityCheckStep,
    DataQualityCheckConfig,
)
from sagemaker.workflow.check_job_config import CheckJobConfig
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator
from sagemaker.model import Model
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker import get_execution_role, image_uris

In [ ]:
session = sagemaker.Session()
region  = session.boto_region_name
role    = get_execution_role()

def tf_output(key):
    r = subprocess.run(
        ['terraform', 'output', '-raw', key],
        cwd='/home/ec2-user/SageMaker/../terraform',
        capture_output=True, text=True, check=True
    )
    return r.stdout.strip()

TRAINING_BUCKET     = tf_output('training_bucket')
ARTIFACTS_BUCKET    = tf_output('artifacts_bucket')
MODEL_PACKAGE_GROUP = tf_output('model_package_group')
SUBNET_ID           = tf_output('private_subnet_id')
PIPELINE_NAME       = tf_output('pipeline_name') + '-abalone'

print(f'Region:              {region}')
print(f'Training bucket:     {TRAINING_BUCKET}')
print(f'Artifacts bucket:    {ARTIFACTS_BUCKET}')
print(f'Model package group: {MODEL_PACKAGE_GROUP}')
print(f'Pipeline name:       {PIPELINE_NAME}')

## 2. Download Abalone dataset

In [ ]:
# Abalone is the SageMaker hello-world dataset
# 4,177 samples, predict age (rings) from physical measurements
# available directly from the UCI ML repository

ABALONE_URL = (
    'https://datahub.io/machine-learning/abalone/r/abalone.csv'
)

os.makedirs('/tmp/abalone', exist_ok=True)
urllib.request.urlretrieve(ABALONE_URL, '/tmp/abalone/abalone.csv')

# upload to S3
s3 = boto3.client('s3')
s3.upload_file(
    '/tmp/abalone/abalone.csv',
    TRAINING_BUCKET,
    'abalone/raw/abalone.csv'
)

print('Abalone dataset uploaded to:')
print(f'  s3://{TRAINING_BUCKET}/abalone/raw/abalone.csv')
print()
print('Dataset: 4,177 samples, 9 features')
print('Target:  rings (continuous — proxy for age)')
print('Task:    regression (predict age from physical measurements)')

## 3. Pipeline parameters

In [ ]:
p_training_bucket  = ParameterString(
    name          = 'TrainingBucket',
    default_value = TRAINING_BUCKET
)
p_artifacts_bucket = ParameterString(
    name          = 'ArtifactsBucket',
    default_value = ARTIFACTS_BUCKET
)

# quality gate — RMSE must be below this threshold
# abalone baseline RMSE with a naive mean predictor ~3.2
# a well-tuned XGBoost model typically achieves RMSE ~2.1
# default threshold 2.5 — reasonable gate between naive and good
# raise to 9.99 to force the FailStep in demos
p_rmse_threshold = ParameterFloat(
    name          = 'RMSEThreshold',
    default_value = 2.5
)

print('Parameters defined:')
print(f'  RMSEThreshold: {p_rmse_threshold.default_value}')
print('  Lower this value to trigger the FailStep in demos.')
print('  Raise above 9.99 to guarantee the FailStep fires.')

## 4. ProcessingStep — feature engineering

In [ ]:
%%writefile preprocess_abalone.py
"""Abalone preprocessing for XGBoost.

Input:  raw abalone.csv
Output: train.csv, validation.csv, test.csv (no headers, label first column)
        baseline.csv (headers, for data quality monitoring)

XGBoost built-in requires:
  - no header row
  - label in the first column
  - all numeric values
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

INPUT_PATH  = '/opt/ml/processing/input'
OUTPUT_PATH = '/opt/ml/processing/output'
os.makedirs(OUTPUT_PATH, exist_ok=True)

df = pd.read_csv(f'{INPUT_PATH}/abalone.csv')
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

# --- encode sex (M/F/I) as numeric ---
# XGBoost built-in does not handle string columns
df['sex'] = df['sex'].map({'M': 0, 'F': 1, 'I': 2})

# --- feature engineering ---
# volume approximation from shell dimensions
df['volume'] = df['length'] * df['diameter'] * df['height']

# weight ratios — shucked weight as fraction of whole weight
df['shucked_ratio'] = df['shucked_weight'] / df['whole_weight'].clip(lower=0.001)

print(f'Features after engineering: {df.shape[1]} columns')

# --- split: 70% train, 15% validation, 15% test ---
target = 'rings'
X = df.drop(columns=[target])
y = df[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# --- write outputs (label first, no header — XGBoost format) ---
def write_split(X, y, name):
    out = pd.concat([y.reset_index(drop=True),
                     X.reset_index(drop=True)], axis=1)
    out.to_csv(f'{OUTPUT_PATH}/{name}.csv', index=False, header=False)
    print(f'  {name}.csv: {len(out):,} rows')

write_split(X_train, y_train, 'train')
write_split(X_val,   y_val,   'validation')
write_split(X_test,  y_test,  'test')

# baseline with headers for data quality monitoring
X_train.to_csv(f'{OUTPUT_PATH}/baseline.csv', index=False)
print(f'  baseline.csv: {len(X_train):,} rows')
print('ProcessingStep complete.')

In [ ]:
sklearn_processor = SKLearnProcessor(
    framework_version = '1.2-1',
    instance_type     = 'ml.m5.large',
    instance_count    = 1,
    role              = role,
    sagemaker_session = session,
    network_config    = sagemaker.network.NetworkConfig(
        enable_network_isolation = False,
        subnets                  = [SUBNET_ID],
    )
)

step_process = ProcessingStep(
    name      = 'ProcessingStep',
    processor = sklearn_processor,
    code      = 'preprocess_abalone.py',
    inputs    = [
        ProcessingInput(
            source      = f's3://{TRAINING_BUCKET}/abalone/raw/abalone.csv',
            destination = '/opt/ml/processing/input',
            input_name  = 'raw-data',
        )
    ],
    outputs   = [
        ProcessingOutput(
            source      = '/opt/ml/processing/output',
            destination = f's3://{ARTIFACTS_BUCKET}/abalone/processing',
            output_name = 'processed-data',
        )
    ]
)

print('ProcessingStep defined.')
print('  Runtime: ~2 minutes')

## 5. TrainingStep — XGBoost

In [ ]:
# XGBoost built-in container — no custom code needed
xgboost_image = image_uris.retrieve(
    framework         = 'xgboost',
    region            = region,
    version           = '1.7-1',
    instance_type     = 'ml.m5.xlarge',
)

xgb_estimator = Estimator(
    image_uri          = xgboost_image,
    instance_type      = 'ml.m5.xlarge',
    instance_count     = 1,
    role               = role,
    sagemaker_session  = session,
    output_path        = f's3://{ARTIFACTS_BUCKET}/abalone/model',

    # VPC config — training job runs in the private subnet
    subnets            = [SUBNET_ID],
    security_group_ids = [],  # populated from terraform output

    hyperparameters = {
        'objective':        'reg:squarederror',
        'num_round':        100,
        'max_depth':        6,
        'eta':              0.2,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'eval_metric':      'rmse',
        'early_stopping_rounds': 10,
    }
)

processed_output = step_process.properties.ProcessingOutputConfig \
    .Outputs['processed-data'].S3Output.S3Uri

step_train = TrainingStep(
    name       = 'TrainingStep',
    estimator  = xgb_estimator,
    inputs     = {
        'train': TrainingInput(
            s3_data       = Join(on='/', values=[processed_output, 'train.csv']),
            content_type  = 'text/csv',
        ),
        'validation': TrainingInput(
            s3_data       = Join(on='/', values=[processed_output, 'validation.csv']),
            content_type  = 'text/csv',
        ),
    },
    depends_on = [step_process]
)

print('TrainingStep defined.')
print('  Model:    XGBoost 1.7-1 (built-in container)')
print('  Runtime:  ~3-5 minutes')
print('  Objective: reg:squarederror — predicting abalone rings (age)')

## 6. QualityCheckStep — RMSE threshold

In [ ]:
# data quality check against the training baseline
# detects distribution shift between training data and future inference inputs

data_quality_check_config = DataQualityCheckConfig(
    baseline_dataset = Join(
        on     = '/',
        values = [processed_output, 'baseline.csv']
    ),
    dataset_format   = DatasetFormat.csv(header=True),
    output_s3_uri    = f's3://{ARTIFACTS_BUCKET}/abalone/quality-check',
)

step_quality = QualityCheckStep(
    name                  = 'QualityCheckStep',
    quality_check_config  = data_quality_check_config,
    check_job_config      = CheckJobConfig(role=role),
    skip_check            = False,
    register_new_baseline = True,
    depends_on            = [step_train]
)

print('QualityCheckStep defined.')
print('  Baseline: training data distribution')
print('  Output:   constraint violations + statistics written to S3')

## 7. ConditionStep — quality gate

In [ ]:
# --- FailStep ---
# fires when RMSE exceeds the threshold
# rejection message includes actual RMSE, threshold, and execution ID
# this record is permanent in CloudWatch and the SageMaker console

step_fail = FailStep(
    name          = 'FailStep',
    error_message = Join(
        on = ' ',
        values = [
            'Model failed quality gate.',
            'RMSE exceeded threshold of',
            p_rmse_threshold,
            '— registration blocked.',
            'Execution:',
            ExecutionVariables.PIPELINE_EXECUTION_ID,
        ]
    )
)

# --- ModelStep (pass branch) ---

xgb_model = Model(
    image_uri         = xgboost_image,
    model_data        = step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session = session,
    role              = role,
)

step_register = ModelStep(
    name      = 'ModelStep',
    step_args = xgb_model.register(
        content_types            = ['text/csv'],
        response_types           = ['text/csv'],
        inference_instances      = ['ml.m5.xlarge'],
        transform_instances      = ['ml.m5.xlarge'],
        model_package_group_name = MODEL_PACKAGE_GROUP,
        approval_status          = 'PendingManualApproval',
        description              = Join(
            on     = ' ',
            values = [
                'XGBoost Abalone.',
                'Execution:',
                ExecutionVariables.PIPELINE_EXECUTION_ID
            ]
        ),
    )
)

# --- ConditionStep ---
# condition: training RMSE <= p_rmse_threshold
# XGBoost reports final validation RMSE in training job metrics

condition_rmse = ConditionLessThanOrEqualTo(
    left  = step_train.properties.FinalMetricDataList[
        'validation:rmse'
    ].Value,
    right = p_rmse_threshold,
)

step_condition = ConditionStep(
    name       = 'ConditionStep',
    conditions = [condition_rmse],
    if_steps   = [step_register],
    else_steps = [step_fail],
    depends_on = [step_quality]
)

print('ConditionStep defined.')
print(f'  Condition: RMSE <= {p_rmse_threshold.default_value}')
print('  Pass → ModelStep (register to Model Registry)')
print('  Fail → FailStep  (log rejection, halt pipeline)')

## 8. Assemble and create the pipeline

In [ ]:
pipeline = Pipeline(
    name       = PIPELINE_NAME,
    parameters = [
        p_training_bucket,
        p_artifacts_bucket,
        p_rmse_threshold,
    ],
    steps      = [
        step_process,
        step_train,
        step_quality,
        step_condition,
    ],
    sagemaker_session = session,
)

pipeline.upsert(role_arn=role)

print(f'Pipeline upserted: {PIPELINE_NAME}')
print()
print('Total expected runtime: ~10 minutes')
print('  ProcessingStep:  ~2 min')
print('  TrainingStep:    ~5 min')
print('  QualityCheck:    ~2 min')
print('  ConditionStep:   ~1 min')

## 9. Run — passing execution

In [ ]:
# default threshold of 2.5 — XGBoost should clear this comfortably
# expected result: ConditionStep routes to ModelStep
# model appears in Model Registry with status PendingManualApproval

execution_pass = pipeline.start(
    parameters = {
        'RMSEThreshold': 2.5,
    },
    execution_display_name = 'passing-run',
    execution_description  = 'XGBoost Abalone — expected to pass quality gate',
)

print(f'Execution started: {execution_pass.arn}')
print()
print('Expected result: PASS')
print('  ConditionStep → ModelStep → model in registry')
print()
print('Monitor in SageMaker Studio:')
print('  Pipelines → ' + PIPELINE_NAME + ' → Executions')

## 10. Run — FailStep demo

In [ ]:
# set threshold below any realistic RMSE to force the FailStep
# XGBoost on Abalone typically achieves RMSE ~2.1
# setting threshold to 1.0 guarantees the FailStep fires
#
# this demonstrates the governance layer in action:
# - ConditionStep evaluates RMSE against threshold
# - RMSE fails the gate
# - FailStep logs: 'Model failed quality gate. RMSE exceeded threshold of 1.0'
# - Pipeline halts with status Failed
# - Execution record is permanent and auditable

execution_fail = pipeline.start(
    parameters = {
        'RMSEThreshold': 1.0,  # nothing will clear this
    },
    execution_display_name = 'failstep-demo',
    execution_description  = 'FailStep demo — threshold set below achievable RMSE',
)

print(f'Execution started: {execution_fail.arn}')
print()
print('Expected result: FAIL')
print('  ConditionStep → FailStep → pipeline halts')
print()
print('Find the rejection record in:')
print('  SageMaker Studio → Pipelines → Executions → failstep-demo → FailStep → Output')
print('  CloudWatch Logs  → /aws/sagemaker/pipelines')
print()
print('The rejection message includes:')
print('  - Threshold value (1.0)')
print('  - Pipeline execution ID')
print('  - Timestamp (from CloudWatch)')
print()
print('This is the auditable rejection record.')
print('Run the cell above (passing execution) side-by-side to show the contrast.')

## 11. Inspect results

In [ ]:
# compare both executions side by side
# run after both executions complete (~10 min each)

sm_client = boto3.client('sagemaker')

for label, arn in [
    ('Passing run', execution_pass.arn),
    ('FailStep demo', execution_fail.arn),
]:
    desc   = sm_client.describe_pipeline_execution(
        PipelineExecutionArn=arn
    )
    status = desc['PipelineExecutionStatus']
    steps  = sm_client.list_pipeline_execution_steps(
        PipelineExecutionArn=arn
    )['PipelineExecutionSteps']

    print(f'{label} — {status}')
    for step in steps:
        name     = step.get('StepName', '—')
        s_status = step.get('StepStatus', '—')
        print(f'  {name:<25} {s_status}')
    print()

# expected output:
#
# Passing run — Succeeded
#   ProcessingStep            Succeeded
#   TrainingStep              Succeeded
#   QualityCheckStep          Succeeded
#   ConditionStep             Succeeded
#   ModelStep                 Succeeded
#
# FailStep demo — Failed
#   ProcessingStep            Succeeded
#   TrainingStep              Succeeded
#   QualityCheckStep          Succeeded
#   ConditionStep             Succeeded
#   FailStep                  Failed

## 12. Next steps

**You've just run a hardened SageMaker Pipeline end to end.**

What you demonstrated:
- Training jobs running in a **private subnet** with no internet gateway
- S3 traffic routing through a **VPC endpoint** — never touches the public internet
- A **QualityCheckStep** that evaluates model performance before anything moves forward
- A **ConditionStep** that enforces a quality threshold as a hard gate
- A **FailStep** that produces an auditable rejection record when a model doesn't qualify
- A model that only reaches the **Model Registry** after clearing the gate

**To deploy the passing model as an endpoint:**
1. Go to SageMaker Studio → Model Registry → approve the model
2. Run `make endpoint` from the terminal

**To run the full production pipeline (fintech credit risk, Autopilot):**
- Open `01_secure_pipeline.ipynb`
- Same infrastructure, same pipeline pattern, real business dataset